# Mirror Test — Kaggle GPU driver

Runs the GPU phases on Kaggle's free **2× T4** (best for the 14B judge; 30 GPU-h/week quota).

**One-time notebook settings (right sidebar):**
1. Session options → Accelerator: **GPU T4 x2**
2. Session options → Internet: **ON** (nothing can download otherwise)
3. Add-ons → Secrets → add `HF_TOKEN` = your Hugging Face *read* token, and attach it to this notebook (needed for the gated Llama/Gemma foils; licenses must be accepted on huggingface.co first)
4. *(optional, for git-push checkpointing)* add a secret `GITHUB_PAT` = a GitHub personal access token with repo scope

**Session rhythm:** run SETUP top-to-bottom, run ONE stage cell, then run the CHECKPOINT cell and *Save Version*. If the session dies mid-run, start again — every script resumes automatically.

**Resuming without GitHub:** each *Save Version* stores `data/` + `results/` as this notebook's output. Next session: attach that output as an Input (+ Add Input → Your Work), and pass its name to `kaggle_setup.py --from-input` in SETUP 3.

In [ ]:
# SETUP 1/3 — get the code + dependencies (Kaggle preinstalls torch)
REPO_URL = "https://github.com/YOUR-USERNAME/mirror-test-llms.git"   # <-- EDIT
import os
if not os.path.exists("/kaggle/working/mirror-test-llms"):
    !cd /kaggle/working && git clone {REPO_URL}
%cd /kaggle/working/mirror-test-llms
!git pull
!pip -q install -U transformers accelerate bitsandbytes datasets sentence-transformers pyyaml

In [ ]:
# SETUP 2/3 — bootstrap: GPUs, HF auth (from the HF_TOKEN secret), writable paths
!python src/kaggle_setup.py

In [ ]:
# SETUP 3/3 — ONLY when resuming from a previous session's saved output
# (skip on your first session). Attach the old version's output as an Input
# first, then put its folder name here:
#!python src/kaggle_setup.py --from-input mirror-test-llms-session2

## STAGE cells — uncomment the line(s) this session is for
Same commands as Colab (`docs/03_pipeline_walkthrough.md` has the full plan). Kaggle's 2×T4 is the right place for everything involving the **14B** models.

In [ ]:
# STAGE A — GENERATION (one model per session)
!python src/01_generate.py --models qwen2.5-14b-instruct --placebo
#!python src/01_generate.py --models gemma-2-9b-it
#!python src/01_generate.py --models mistral-7b-instruct-v0.3

In [ ]:
# STAGE B — pairs (CPU-fast, after ALL generation) + paraphrase materials (GPU)
#!python src/02_build_pairs.py
#!python src/02b_paraphrase.py

In [ ]:
# STAGE C — judging
#!python src/03_judge_ppp.py --judge qwen2.5-14b-instruct --include-placebo
#!python src/03_judge_ppp.py --judge qwen2.5-14b-instruct --foils llama-3.2-3b-instruct --phrasings 0 1 2 --include-paraphrase
#!python src/04_judge_ipp.py --judge qwen2.5-14b-instruct
#!python src/03_judge_ppp.py --judge qwen2.5-14b-base
#!python src/05_baselines.py perplexity --judge qwen2.5-14b-instruct --include-paraphrase

In [ ]:
# CHECKPOINT — ALWAYS run before ending the session, then click 'Save Version'.
# Option 1 (no GitHub): Save Version alone preserves /kaggle/working as output.
!du -sh data results 2>/dev/null; ls results/judgments/ 2>/dev/null

# Option 2 (git push, needs the GITHUB_PAT secret):
#from kaggle_secrets import UserSecretsClient
#import subprocess
#pat = UserSecretsClient().get_secret("GITHUB_PAT")
#url = REPO_URL.replace("https://", f"https://x-access-token:{pat}@")
#!git config user.email "you@example.com" && git config user.name "Your Name"
#!git add -A && git commit -m "kaggle session results" || echo 'nothing new'
#subprocess.run(["git", "push", url, "HEAD:main"])  # keeps the token out of shell logs